# Case 1 Demo: FNO, PINO, PINN

Single notebook for the case 1 comparison.

- shared random seed
- shared train/test split
- train-only normalization for FNO and PINO
- same validation and test files
- same finite-difference PDE metric

One split and one evaluation pipeline are used throughout.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import math
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

try:
    import deepxde as dde
except ImportError as exc:
    raise ImportError(
        "This notebook requires deepxde. Install dependencies from requirements.txt before running."
    ) from exc

try:
    from neuralop.models import FNO
except ImportError as exc:
    raise ImportError(
        "This notebook requires neuraloperator/neuralop. Install dependencies from requirements.txt before running."
    ) from exc


@dataclass
class FairConfig:
    train_npz: str = "train_20x20x20x20_Ngeom.npz"
    val_npz: str = "val_20x20x19x19_Ngeom.npz"
    test_mild_npz: str = "test_mild_10_20x20x20x20_Ngeom.npz"
    test_ood_npz: str = "test_phi2_ood_right_20x20x20x20_Ngeom.npz"
    output_dir: str = "case1_outputs"
    split_seed: int = 0
    train_fraction: float = 0.9
    train_batch_size: int = 16
    eval_batch_size: int = 32
    operator_epochs: int = 250
    pino_pretrain_fraction: float = 0.48
    pino_lambda_pde: float = 3e-4
    pinn_stage1_fraction: float = 0.4
    pinn_lambda_pde: float = 1e-6
    pinn_lambda_bc: float = 1.0
    pinn_lambda_ic: float = 1.0
    pinn_lambda_data: float = 1.0
    pinn_num_domain: int = 20000
    pinn_num_boundary: int = 2000
    eval_pde_subset: int = 50000
    reaction_order: int = 2
    fno_n_modes: tuple[int, int] = (8, 8)
    fno_hidden_channels: int = 32
    pinn_hidden_width: int = 128
    pinn_hidden_depth: int = 6


CFG = FairConfig()
OUTPUT_DIR = Path(CFG.output_dir)
OUTPUT_DIR.mkdir(exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())


def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        dde.config.set_random_seed(seed)
    except Exception:
        pass


set_global_seed(CFG.split_seed)


In [ ]:
def load_case1_dataset(npz_path: str) -> dict[str, np.ndarray]:
    data = np.load(npz_path)
    U = data["U"].astype(np.float32)
    tau = data["tau"].astype(np.float32)
    x = data["x"].astype(np.float32)
    phi2 = data["phi2"].astype(np.float32)
    Nval = data["N"].astype(np.float32)
    return {"U": U, "tau": tau, "x": x, "phi2": phi2, "N": Nval}


def build_operator_samples(data: dict[str, np.ndarray]) -> tuple[np.ndarray, np.ndarray]:
    U, tau, x, phi2, Nval = data["U"], data["tau"], data["x"], data["phi2"], data["N"]
    nt, nx, nphi, nN = U.shape
    T, Xgrid = np.meshgrid(tau, x, indexing="ij")

    nsamples = nphi * nN
    X_all = np.zeros((nsamples, 4, nt, nx), dtype=np.float32)
    Y_all = np.zeros((nsamples, 1, nt, nx), dtype=np.float32)

    idx = 0
    for j in range(nN):
        for k in range(nphi):
            X_all[idx, 0] = T
            X_all[idx, 1] = Xgrid
            X_all[idx, 2] = phi2[k]
            X_all[idx, 3] = Nval[j]
            Y_all[idx, 0] = U[:, :, k, j]
            idx += 1
    return X_all, Y_all


def build_coordinate_samples(data: dict[str, np.ndarray]) -> tuple[np.ndarray, np.ndarray]:
    U, tau, x, phi2, Nval = data["U"], data["tau"], data["x"], data["phi2"], data["N"]
    nt, nx, nphi, nN = U.shape
    T, Xgrid = np.meshgrid(tau, x, indexing="ij")
    tau_flat = T.reshape(-1, 1).astype(np.float32)
    x_flat = Xgrid.reshape(-1, 1).astype(np.float32)
    P = nt * nx

    nsamples = nphi * nN
    X_samps = np.zeros((nsamples, P, 4), dtype=np.float32)
    Y_samps = np.zeros((nsamples, P, 1), dtype=np.float32)

    idx = 0
    for j in range(nN):
        for k in range(nphi):
            phi_col = np.full_like(tau_flat, phi2[k], dtype=np.float32)
            N_col = np.full_like(tau_flat, Nval[j], dtype=np.float32)
            X_samps[idx] = np.concatenate([tau_flat, x_flat, phi_col, N_col], axis=1)
            Y_samps[idx] = U[:, :, k, j].reshape(-1, 1).astype(np.float32)
            idx += 1
    return X_samps, Y_samps


train_data = load_case1_dataset(CFG.train_npz)
X_operator_all, Y_operator_all = build_operator_samples(train_data)
X_coord_all, Y_coord_all = build_coordinate_samples(train_data)

nsamples = X_operator_all.shape[0]
rng = np.random.default_rng(CFG.split_seed)
perm = rng.permutation(nsamples)
train_size = int(CFG.train_fraction * nsamples)
tr_idx = perm[:train_size]
te_idx = perm[train_size:]

np.savez(
    OUTPUT_DIR / "case1_split_indices.npz",
    tr_idx=tr_idx,
    te_idx=te_idx,
    seed=np.array([CFG.split_seed], dtype=np.int64),
)

print("Total samples:", nsamples)
print("Train samples:", len(tr_idx))
print("Test samples:", len(te_idx))


In [ ]:
# Train-only normalization for operator models.
X_mean = X_operator_all[tr_idx].mean(axis=(0, 2, 3), keepdims=True)
X_std = X_operator_all[tr_idx].std(axis=(0, 2, 3), keepdims=True) + 1e-12
Y_mean = Y_operator_all[tr_idx].mean()
Y_std = Y_operator_all[tr_idx].std() + 1e-12

X_operator_norm = (X_operator_all - X_mean) / X_std
Y_operator_norm = (Y_operator_all - Y_mean) / Y_std

X_tr = X_operator_norm[tr_idx]
Y_tr = Y_operator_norm[tr_idx]
X_te = X_operator_norm[te_idx]
Y_te = Y_operator_norm[te_idx]

X_tr_t = torch.tensor(X_tr, dtype=torch.float32)
Y_tr_t = torch.tensor(Y_tr, dtype=torch.float32)
X_te_t = torch.tensor(X_te, dtype=torch.float32)
Y_te_t = torch.tensor(Y_te, dtype=torch.float32)

g_cpu = torch.Generator(device="cpu")
g_cpu.manual_seed(CFG.split_seed)

train_loader = DataLoader(
    TensorDataset(X_tr_t, Y_tr_t),
    batch_size=CFG.train_batch_size,
    shuffle=True,
    generator=g_cpu,
    num_workers=0,
)
test_loader = DataLoader(
    TensorDataset(X_te_t, Y_te_t),
    batch_size=CFG.eval_batch_size,
    shuffle=False,
    num_workers=0,
)

operator_updates = CFG.operator_epochs * len(train_loader)
pino_pretrain_epochs = max(1, int(round(CFG.operator_epochs * CFG.pino_pretrain_fraction)))
pinn_total_iters = operator_updates
pinn_stage1_iters = max(1, int(round(pinn_total_iters * CFG.pinn_stage1_fraction)))
pinn_stage2_iters = max(1, pinn_total_iters - pinn_stage1_iters)

print("Operator updates per model:", operator_updates)
print("PINO pretrain epochs:", pino_pretrain_epochs)
print("PINN stage1 iterations:", pinn_stage1_iters)
print("PINN stage2 iterations:", pinn_stage2_iters)


In [ ]:
def build_operator_model() -> nn.Module:
    return FNO(
        n_modes=CFG.fno_n_modes,
        hidden_channels=CFG.fno_hidden_channels,
        in_channels=4,
        out_channels=1,
    ).to(DEVICE)


def denorm_operator(pred_norm: torch.Tensor) -> torch.Tensor:
    y_mean_t = torch.tensor(float(Y_mean), device=pred_norm.device, dtype=pred_norm.dtype)
    y_std_t = torch.tensor(float(Y_std), device=pred_norm.device, dtype=pred_norm.dtype)
    return pred_norm * y_std_t + y_mean_t


def operator_pde_loss(pred_norm: torch.Tensor, xb_norm: torch.Tensor, eps_x: float = 1e-6) -> torch.Tensor:
    u = denorm_operator(pred_norm)

    xm = torch.tensor(X_mean, device=pred_norm.device, dtype=pred_norm.dtype)
    xs = torch.tensor(X_std, device=pred_norm.device, dtype=pred_norm.dtype)
    tau = xb_norm[:, 0:1] * xs[:, 0:1] + xm[:, 0:1]
    x = xb_norm[:, 1:2] * xs[:, 1:2] + xm[:, 1:2]
    phi2 = xb_norm[:, 2:3] * xs[:, 2:3] + xm[:, 2:3]
    N = xb_norm[:, 3:4] * xs[:, 3:4] + xm[:, 3:4]

    dtau = (tau[:, :, 1:, :].mean() - tau[:, :, :-1, :].mean()).abs() + 1e-12
    dx = (x[:, :, :, 1:].mean() - x[:, :, :, :-1].mean()).abs() + 1e-12

    u_tau = torch.zeros_like(u)
    u_tau[:, :, 1:-1, :] = (u[:, :, 2:, :] - u[:, :, :-2, :]) / (2.0 * dtau)
    u_tau[:, :, 0, :] = (u[:, :, 1, :] - u[:, :, 0, :]) / dtau
    u_tau[:, :, -1, :] = (u[:, :, -1, :] - u[:, :, -2, :]) / dtau

    u_x = torch.zeros_like(u)
    u_x[:, :, :, 1:-1] = (u[:, :, :, 2:] - u[:, :, :, :-2]) / (2.0 * dx)
    u_x[:, :, :, 0] = (u[:, :, :, 1] - u[:, :, :, 0]) / dx
    u_x[:, :, :, -1] = (u[:, :, :, -1] - u[:, :, :, -2]) / dx

    u_xx = torch.zeros_like(u)
    u_xx[:, :, :, 1:-1] = (u[:, :, :, 2:] - 2 * u[:, :, :, 1:-1] + u[:, :, :, :-2]) / (dx * dx)
    u_xx[:, :, :, 0] = u_xx[:, :, :, 1]
    u_xx[:, :, :, -1] = u_xx[:, :, :, -2]

    Nx = N / (x + eps_x)
    residual = u_tau - u_xx - Nx * u_x + phi2 * (u ** CFG.reaction_order)
    return torch.mean(residual[:, :, :, 1:-1] ** 2)


def train_fno() -> tuple[nn.Module, dict[str, float]]:
    set_global_seed(CFG.split_seed)
    model = build_operator_model()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()

    final_train_loss = float("nan")
    final_test_loss = float("nan")
    for ep in range(1, CFG.operator_epochs + 1):
        model.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * xb.size(0)
        train_loss /= len(train_loader.dataset)

        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)
                pred = model(xb)
                loss = criterion(pred, yb)
                test_loss += loss.item() * xb.size(0)
        test_loss /= len(test_loader.dataset)

        final_train_loss = train_loss
        final_test_loss = test_loss
        if ep % 25 == 0 or ep == 1:
            print(f"[FNO] epoch {ep:4d}/{CFG.operator_epochs} | train {train_loss:.3e} | test {test_loss:.3e}")

    ckpt_path = OUTPUT_DIR / "case1_fno_2d_phi2N_to_u.th"
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "split_seed": CFG.split_seed,
            "tr_idx": tr_idx,
            "te_idx": te_idx,
            "x_mean": X_mean,
            "x_std": X_std,
            "y_mean": float(Y_mean),
            "y_std": float(Y_std),
            "config": vars(CFG),
        },
        ckpt_path,
    )
    print("Saved FNO checkpoint to:", ckpt_path)
    return model, {"train_loss": final_train_loss, "test_loss": final_test_loss}


def train_pino() -> tuple[nn.Module, dict[str, float]]:
    set_global_seed(CFG.split_seed)
    model = build_operator_model()
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    final_train_loss = float("nan")
    final_test_loss = float("nan")
    final_train_data = float("nan")
    final_train_pde = float("nan")

    for ep in range(1, CFG.operator_epochs + 1):
        if ep == pino_pretrain_epochs + 1:
            optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
            print("\n[PINO] switching to AdamW(lr=1e-4, wd=1e-4) and enabling PDE loss\n")

        model.train()
        train_loss = 0.0
        train_data = 0.0
        train_pde = 0.0
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            pred = model(xb)
            loss_data = criterion(pred, yb)
            if ep > pino_pretrain_epochs:
                loss_pde = operator_pde_loss(pred, xb)
                loss = loss_data + CFG.pino_lambda_pde * loss_pde
            else:
                loss_pde = torch.zeros((), device=DEVICE)
                loss = loss_data
            loss.backward()
            optimizer.step()

            bs = xb.size(0)
            train_loss += loss.item() * bs
            train_data += loss_data.item() * bs
            train_pde += loss_pde.item() * bs

        train_loss /= len(train_loader.dataset)
        train_data /= len(train_loader.dataset)
        train_pde /= len(train_loader.dataset)

        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)
                pred = model(xb)
                loss = criterion(pred, yb)
                test_loss += loss.item() * xb.size(0)
        test_loss /= len(test_loader.dataset)

        final_train_loss = train_loss
        final_test_loss = test_loss
        final_train_data = train_data
        final_train_pde = train_pde
        if ep % 25 == 0 or ep == 1:
            print(
                f"[PINO] epoch {ep:4d}/{CFG.operator_epochs} | "
                f"train {train_loss:.3e} | data {train_data:.3e} | pde {train_pde:.3e} | test {test_loss:.3e}"
            )

    ckpt_path = OUTPUT_DIR / "case1_pino_2d_phi2N_to_u.th"
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "split_seed": CFG.split_seed,
            "tr_idx": tr_idx,
            "te_idx": te_idx,
            "x_mean": X_mean,
            "x_std": X_std,
            "y_mean": float(Y_mean),
            "y_std": float(Y_std),
            "config": vars(CFG),
        },
        ckpt_path,
    )
    print("Saved PINO checkpoint to:", ckpt_path)
    return model, {
        "train_loss": final_train_loss,
        "test_loss": final_test_loss,
        "train_data_loss": final_train_data,
        "train_pde_loss": final_train_pde,
    }


In [ ]:
def build_pinn_train_arrays(
    X_samps: np.ndarray,
    Y_samps: np.ndarray,
    tr_idx: np.ndarray,
    te_idx: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    X_train = X_samps[tr_idx].reshape(-1, 4)
    y_train = Y_samps[tr_idx].reshape(-1, 1)
    X_test = X_samps[te_idx].reshape(-1, 4)
    y_test = Y_samps[te_idx].reshape(-1, 1)
    return X_train, y_train, X_test, y_test


def train_pinn() -> tuple[dde.Model, dict[str, float]]:
    set_global_seed(CFG.split_seed)
    X_train, y_train, X_test, y_test = build_pinn_train_arrays(X_coord_all, Y_coord_all, tr_idx, te_idx)

    tau = train_data["tau"]
    x = train_data["x"]
    phi2 = train_data["phi2"]
    Nval = train_data["N"]

    tau_min, tau_max = float(tau.min()), float(tau.max())
    x_min_data, x_max = float(x.min()), float(x.max())
    phi_min, phi_max = float(phi2.min()), float(phi2.max())
    N_min, N_max = float(Nval.min()), float(Nval.max())
    x_min_pde = max(1e-5, x_min_data)

    geom = dde.geometry.Hypercube(
        [tau_min, x_min_pde, phi_min, N_min],
        [tau_max, x_max, phi_max, N_max],
    )

    def pde(X, u):
        u_tau = dde.grad.jacobian(u, X, i=0, j=0)
        u_x = dde.grad.jacobian(u, X, i=0, j=1)
        u_xx = dde.grad.hessian(u, X, component=0, i=1, j=1)
        x_var = X[:, 1:2]
        phi2_var = X[:, 2:3]
        N_var = X[:, 3:4]
        Nx = N_var / (x_var + 1e-12)
        diffusion = u_xx + Nx * u_x
        reaction = phi2_var * (u ** CFG.reaction_order)
        return u_tau - diffusion + reaction

    def on_x1_boundary(X, on_boundary):
        return on_boundary and np.isclose(X[1], x_max)

    def on_x0_boundary(X, on_boundary):
        return on_boundary and np.isclose(X[1], x_min_pde)

    def on_tau0_boundary(X, on_boundary):
        return on_boundary and np.isclose(X[0], tau_min)

    bc_x1 = dde.icbc.DirichletBC(geom, lambda X: 1.0, on_x1_boundary)
    bc_x0 = dde.icbc.NeumannBC(geom, lambda X: 0.0, on_x0_boundary)
    ic_t0 = dde.icbc.DirichletBC(geom, lambda X: 1.0, on_tau0_boundary)
    bc_data = dde.icbc.PointSetBC(X_train, y_train, component=0)

    data_pinn = dde.data.PDE(
        geom,
        pde,
        bcs=[bc_x1, bc_x0, ic_t0, bc_data],
        num_domain=CFG.pinn_num_domain,
        num_boundary=CFG.pinn_num_boundary,
    )

    net = dde.maps.FNN(
        [4] + [CFG.pinn_hidden_width] * CFG.pinn_hidden_depth + [1],
        "tanh",
        "Glorot normal",
    )
    model = dde.Model(data_pinn, net)

    class TrainTestMSECallback(dde.callbacks.Callback):
        def __init__(self, X_train, y_train, X_test, y_test, period=200):
            super().__init__()
            self.X_train = X_train
            self.y_train = y_train
            self.X_test = X_test
            self.y_test = y_test
            self.period = period

        def on_epoch_end(self):
            it = self.model.train_state.epoch
            if it % self.period == 0:
                y_pred_tr = self.model.predict(self.X_train)
                y_pred_te = self.model.predict(self.X_test)
                tr_mse = np.mean((y_pred_tr - self.y_train) ** 2)
                te_mse = np.mean((y_pred_te - self.y_test) ** 2)
                print(f"[PINN custom] iter {it:6d} | train_mse {tr_mse:.3e} | test_mse {te_mse:.3e}")

    mse_cb = TrainTestMSECallback(X_train, y_train, X_test, y_test, period=200)

    loss_weights = [
        CFG.pinn_lambda_pde,
        CFG.pinn_lambda_bc,
        CFG.pinn_lambda_bc,
        CFG.pinn_lambda_ic,
        CFG.pinn_lambda_data,
    ]
    loss_weights_stage1 = [
        0.0,
        CFG.pinn_lambda_bc,
        CFG.pinn_lambda_bc,
        CFG.pinn_lambda_ic,
        CFG.pinn_lambda_data,
    ]

    model.compile("adam", lr=1e-3, loss_weights=loss_weights_stage1)
    print("\n=== PINN stage 1: data + BC/IC ===")
    model.train(
        iterations=pinn_stage1_iters,
        display_every=200,
        callbacks=[mse_cb],
    )

    model.compile("adam", lr=5e-5, loss_weights=loss_weights)
    print("\n=== PINN stage 2: data + BC/IC + PDE ===")
    model.train(
        iterations=pinn_stage2_iters,
        display_every=200,
        callbacks=[mse_cb],
    )

    y_pred_tr = model.predict(X_train)
    y_pred_te = model.predict(X_test)
    train_mse = float(np.mean((y_pred_tr - y_train) ** 2))
    test_mse = float(np.mean((y_pred_te - y_test) ** 2))

    ckpt_path = OUTPUT_DIR / "case1_pinn_deepxde_phi2N.th"
    torch.save(
        {
            "model_state_dict": model.net.state_dict(),
            "split_seed": CFG.split_seed,
            "tr_idx": tr_idx,
            "te_idx": te_idx,
            "hidden_layers": [CFG.pinn_hidden_width] * CFG.pinn_hidden_depth,
            "config": vars(CFG),
        },
        ckpt_path,
    )
    print("Saved PINN checkpoint to:", ckpt_path)
    return model, {"train_loss": train_mse, "test_loss": test_mse}


def count_parameters_torch(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


fno_model, fno_train_stats = train_fno()
pino_model, pino_train_stats = train_pino()
pinn_model, pinn_train_stats = train_pinn()

model_summary = [
    {"model": "FNO", "train_loss": fno_train_stats["train_loss"], "test_loss": fno_train_stats["test_loss"], "trainable_params": count_parameters_torch(fno_model)},
    {"model": "PINO", "train_loss": pino_train_stats["train_loss"], "test_loss": pino_train_stats["test_loss"], "trainable_params": count_parameters_torch(pino_model)},
    {"model": "PINN", "train_loss": pinn_train_stats["train_loss"], "test_loss": pinn_train_stats["test_loss"], "trainable_params": count_parameters_torch(pinn_model.net)},
]

print("\nTraining summary:")
for row in model_summary:
    print(row)


In [ ]:
def build_operator_eval_inputs(data: dict[str, np.ndarray]) -> tuple[np.ndarray, np.ndarray]:
    X_all, Y_all = build_operator_samples(data)
    X_norm = (X_all - X_mean) / X_std
    return X_norm, Y_all


def operator_predict_grid(model: nn.Module, data: dict[str, np.ndarray]) -> np.ndarray:
    X_norm, Y_all = build_operator_eval_inputs(data)
    loader = DataLoader(
        TensorDataset(torch.tensor(X_norm, dtype=torch.float32), torch.tensor(Y_all, dtype=torch.float32)),
        batch_size=CFG.eval_batch_size,
        shuffle=False,
        num_workers=0,
    )

    preds = []
    model.eval()
    with torch.no_grad():
        for xb, _ in loader:
            xb = xb.to(DEVICE)
            pred_norm = model(xb)
            pred_phys = denorm_operator(pred_norm).cpu().numpy()
            preds.append(pred_phys)
    return np.concatenate(preds, axis=0)[:, 0]


def pinn_predict_grid(model: dde.Model, data: dict[str, np.ndarray]) -> np.ndarray:
    X_samps, _ = build_coordinate_samples(data)
    preds = model.predict(X_samps.reshape(-1, 4)).reshape(X_samps.shape[0], -1)
    nt = data["tau"].shape[0]
    nx = data["x"].shape[0]
    return preds.reshape(X_samps.shape[0], nt, nx).astype(np.float32)


def finite_difference_pde_mse(
    u_pred: np.ndarray,
    tau: np.ndarray,
    x: np.ndarray,
    phi2: np.ndarray,
    Nval: np.ndarray,
    reaction_order: int = 2,
    subset_size: int | None = None,
    seed: int = 0,
) -> float:
    nsamples, nt, nx = u_pred.shape
    dtau = float(np.abs(tau[1] - tau[0])) if nt > 1 else 1.0
    dx = float(np.abs(x[1] - x[0])) if nx > 1 else 1.0

    residuals = []
    idx = 0
    for n_val in Nval:
        for phi_val in phi2:
            u = u_pred[idx]
            idx += 1

            u_tau = np.zeros_like(u)
            u_tau[1:-1, :] = (u[2:, :] - u[:-2, :]) / (2.0 * dtau)
            u_tau[0, :] = (u[1, :] - u[0, :]) / dtau
            u_tau[-1, :] = (u[-1, :] - u[-2, :]) / dtau

            u_x = np.zeros_like(u)
            u_x[:, 1:-1] = (u[:, 2:] - u[:, :-2]) / (2.0 * dx)
            u_x[:, 0] = (u[:, 1] - u[:, 0]) / dx
            u_x[:, -1] = (u[:, -1] - u[:, -2]) / dx

            u_xx = np.zeros_like(u)
            u_xx[:, 1:-1] = (u[:, 2:] - 2 * u[:, 1:-1] + u[:, :-2]) / (dx * dx)
            u_xx[:, 0] = u_xx[:, 1]
            u_xx[:, -1] = u_xx[:, -2]

            Nx = float(n_val) / (x[None, :] + 1e-6)
            residual = u_tau - u_xx - Nx * u_x + float(phi_val) * (u ** reaction_order)
            residuals.append(residual[:, 1:-1].reshape(-1))

    residuals_flat = np.concatenate(residuals, axis=0)
    if subset_size is not None and subset_size < residuals_flat.size:
        rng = np.random.default_rng(seed)
        take = rng.choice(residuals_flat.size, size=subset_size, replace=False)
        residuals_flat = residuals_flat[take]
    return float(np.mean(residuals_flat ** 2))


def evaluate_model_on_dataset(model_name: str, model_obj, data_path: str) -> dict[str, float]:
    data = load_case1_dataset(data_path)
    U_true = data["U"]
    Y_true = build_operator_samples(data)[1][:, 0]

    if model_name in {"FNO", "PINO"}:
        Y_pred = operator_predict_grid(model_obj, data)
    elif model_name == "PINN":
        Y_pred = pinn_predict_grid(model_obj, data)
    else:
        raise ValueError(model_name)

    mse_phys = float(np.mean((Y_pred - Y_true) ** 2))
    pde_mse = finite_difference_pde_mse(
        Y_pred,
        data["tau"],
        data["x"],
        data["phi2"],
        data["N"],
        reaction_order=CFG.reaction_order,
        subset_size=CFG.eval_pde_subset,
        seed=CFG.split_seed,
    )
    return {
        "model": model_name,
        "dataset": Path(data_path).name,
        "mse_phys": mse_phys,
        "pde_mse": pde_mse,
    }


eval_paths = [CFG.val_npz, CFG.test_mild_npz, CFG.test_ood_npz]
all_eval_rows = []
for data_path in eval_paths:
    all_eval_rows.append(evaluate_model_on_dataset("FNO", fno_model, data_path))
    all_eval_rows.append(evaluate_model_on_dataset("PINO", pino_model, data_path))
    all_eval_rows.append(evaluate_model_on_dataset("PINN", pinn_model, data_path))

print("\nEvaluation summary:")
for row in all_eval_rows:
    print(row)

np.savez(
    OUTPUT_DIR / "case1_comparison_metrics.npz",
    rows=np.array(all_eval_rows, dtype=object),
)
print("Saved evaluation metrics to:", OUTPUT_DIR / "case1_comparison_metrics.npz")


In [ ]:
# Lightweight visualization of the physics MSE by dataset.
datasets = [Path(p).name for p in eval_paths]
model_names = ["FNO", "PINO", "PINN"]
bar_width = 0.22
xpos = np.arange(len(datasets))

plt.figure(figsize=(10, 4.5))
for i, model_name in enumerate(model_names):
    vals = [
        next(row["mse_phys"] for row in all_eval_rows if row["model"] == model_name and row["dataset"] == ds)
        for ds in datasets
    ]
    plt.bar(xpos + i * bar_width, vals, width=bar_width, label=model_name)

plt.xticks(xpos + bar_width, datasets, rotation=15)
plt.yscale("log")
plt.ylabel("Physics MSE")
plt.title("Case 1 demo")
plt.legend()
plt.tight_layout()

fig_path = OUTPUT_DIR / "case1_mse_phys_summary.png"
plt.savefig(fig_path, dpi=180)
plt.show()
print("Saved summary figure to:", fig_path)


## Notes

- FNO and PINO share the same operator setup.
- Common split and common evaluation are used throughout.
- Outputs are written to `case1_outputs/`.
